#  SETUP 

**Purpose**
- Validate modularized code in fresh environment (no legacy variables)
- Compare with Shiu et al. (2024) original results
- Serve as template for cloud deployment scripts

---


## 1 Environment Setup

### 1.1 Libraries
**Standard Libraries**
- numpy, pandas, pathlib
- time, pickle, json
- os, sys

**Scientific Computing**
- matplotlib.pyplot
- scipy.stats (pearsonr)
- seaborn

**Brian2 Simulator**
- from brian2 import *
- 或显式：NeuronGroup, Synapses, Network, Hz, ms, mV

**Parallel & Monitoring**
- from joblib import Parallel, delayed
- import psutil, gc

In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from time import time
import pickle

import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import seaborn as sns

from brian2 import Hz, ms

from joblib import Parallel, delayed
import gc
print("✅ Modules imported successfully")

### 1.2 FlyWire Tools
- import pyarrow.parquet as pq
- from caveclient import CAVEclient

In [ ]:
import pyarrow.parquet as pq
from caveclient import CAVEclient
print("Initializing CAVEclient...")
client = CAVEclient('flywire_fafb_public')
print("✅ Connected to FlyWire")

### 1.3 FlyLIF Modules
- from flylif.core import ...
- from flylif.utils import ...

In [ ]:
from flylif.core.parameters import DEFAULT_PARAMS
from flylif.core.data_loader import load_simulation_data
from flylif.core.network import build_network
from flylif.core.simulation import run_simulation
from flylif.utils.visualization import plot_correlation, plot_response_heatmap, plot_frequency_response_curve, plot_summary_statistics
from flylif.utils.cave_utils import convert_neuron_list_cached
from flylif.utils.checkpoint import CheckpointManager
from flylif.utils.memory_utils import print_memory, check_memory_safe, memory_cleanup, MemoryMonitor
print("✅ FlyLIF modules imported")

### 1.4 Auto-reload (development)
- %load_ext autoreload
- %autoreload 2

In [ ]:
%load_ext autoreload
%autoreload 2
print("✅ Auto-reload enabled: .py files will auto-refresh")

## 2 Configuration
### 2.1 Paths and data config

In [ ]:
# Base directory
BASE_DIR = Path.home() / 'mylibrary'/'connectome'/'lifmodel'

# %%
CONFIG = {
    # 基础目录
    'base_dir': BASE_DIR,
    
    # FlyWire 下载的数据文件
    'data_dir': BASE_DIR / 'data_783',
    'connections_file': 'proofread_connections_783.feather',
    'root_ids_file': 'proofread_root_ids_783.npy',
    'classification': 'classification.csv',}


### 2.2 Neuron IDs definition

In [ ]:
# ==================== Cell 新增B: 定义原文v630神经元IDs ====================
print("\n" + "=" * 70)
print("Defining Original Paper Neuron IDs (v630)")
print("=" * 70)

# Original paper neurons (from Shiu et al., FlyWire v630)
NEU_SUGAR_v630 = [
    720575940624963786, 720575940630233916, 720575940637568838, 720575940638202345, 720575940617000768,
    720575940630797113, 720575940632889389, 720575940621754367, 720575940621502051, 720575940640649691,
    720575940639332736, 720575940616885538, 720575940639198653, 720575940620900446, 720575940617937543,
    720575940632425919, 720575940633143833, 720575940612670570, 720575940628853239, 720575940629176663,
    720575940611875570
]

NEU_MN9_v630 = [720575940660219265]  # MN9可能版本间稳定
NEU_MN9_LEFT_v630 = [720575940645521262]  #720575940618238523

print(f"\nOriginal neuron counts:")
print(f"  Sugar GRNs: {len(NEU_SUGAR_v630)}")
print(f"  MN9: {len(NEU_MN9_v630)}")
print(f"  MN9_LEFT: {len(NEU_MN9_LEFT_v630)}")

In [ ]:
# ==================== Cell 新增C: 预转换Sugar GRN IDs（缓存）====================
print("\n" + "=" * 70)
print("Converting Sugar GRN IDs (v630 → v783)")
print("=" * 70)
print("⚠️  This uses FlyWire API and may take 1-2 minutes")
print("   Results will be cached to avoid repeated calls\n")

# Cache file
cache_dir = Path('./cache/id_conversions')
cache_dir.mkdir(parents=True, exist_ok=True)
cache_file = cache_dir / 'sugar_grns_v630_to_v783.pkl'

t0 = time()

# Convert with caching
NEU_SUGAR_LEFT = convert_neuron_list_cached(
    old_list=NEU_SUGAR_v630,
    cache_file=cache_file,
    new_ver=783,
    force_update=False,  # Use cache if available
    verbose=True
)

conversion_time = time() - t0

print(f"\n✅ Conversion complete")
print(f"   Time: {conversion_time:.1f}s")
print(f"   Converted IDs: {len(NEU_SUGAR_LEFT)}")

# Show mapping
print(f"\n   ID mapping (first 5):")
for old, new in zip(NEU_SUGAR_v630[:5], NEU_SUGAR_LEFT[:5]):
    match = "✓" if old == new else "✗ CHANGED"
    print(f"   {old} → {new} {match}")

In [ ]:
# ==================== Cell 新增D: 转换MN9（可选）====================
print("\nConverting MN9 IDs...")

cache_file_mn9 = cache_dir / 'mn9_v630_to_v783.pkl'
cache_file_mn9_left = cache_dir / 'mn9_left_v630_to_v783.pkl'

NEU_MN9_RIGHT = convert_neuron_list_cached(
    old_list=NEU_MN9_v630,
    cache_file=cache_file_mn9,
    new_ver=783,
    verbose=True
)
NEU_MN9_LEFT = convert_neuron_list_cached(
    old_list=NEU_MN9_LEFT_v630,
    cache_file=cache_file_mn9_left,
    new_ver=783,
    verbose=True
)

print(f"   MN9: {NEU_MN9_v630[0]} → {NEU_MN9_RIGHT[0]}")
print(f"   MN9 Left: {NEU_MN9_LEFT_v630[0]} → {NEU_MN9_LEFT[0]}")

In [ ]:
# ==================== Cell 新增E: 保存转换结果（备份）====================
# 保存转换后的神经元列表供后续使用
neuron_ids_v783 = {
    'NEU_SUGAR_LEFT': NEU_SUGAR_LEFT,
    'NEU_MN9_RIGHT': NEU_MN9_RIGHT,
    'NEU_MN9_LEFT':NEU_MN9_LEFT,
    'conversion_time': conversion_time,
    'source_version': 630,
    'target_version': 783,
}

backup_file = cache_dir / 'neuron_ids_v783.pkl'
with open(backup_file, 'wb') as f:
    pickle.dump(neuron_ids_v783, f)

print(f"\n✅ Neuron IDs ready for simulation")
print(f"   Backup saved: {backup_file}")
print(f"\n" + "=" * 70)
print("⚠️  Important: Use NEU_SUGAR_LEFT (v783) for simulation")
print("=" * 70)

### 2.3 EXP1 Batch configuration (4×5)

In [ ]:
# 定义4批，每批5个频率
BATCH_CONFIG = {
    'batch1': [10, 20, 30, 40, 50],
    'batch2': [60, 70, 80, 90, 100],
    'batch3': [110, 120, 130, 140, 150],
    'batch4': [160, 170, 180, 190, 200],
}

## 3 Data Loading
### 3.1 Load optimized data

In [ ]:
print("Loading data...")
t0 = time()

DATA = load_simulation_data(CONFIG)

print(f"✅ Loaded in {time()-t0:.1f}s")
print(f"   Data size: {DATA['df_conn'].memory_usage(deep=True).sum()/1e6:.0f} MB")

# Experiment 1 

## 4 Batched Execution

### 4.1 Checkpoint & monitoring imports

In [ ]:
# ==================== Cell 6A: 导入checkpoint和监控工具 ====================
# 初始检查
print("\n初始内存状态:")
print_memory("  ")

### 4.2 Worker function (with cleanup)

In [ ]:
# ==================== Cell 6B: Worker函数（带内存清理）====================
def run_freq_worker_clean(freq, neu_exc, data, params, n_trials):
    """
    Worker with explicit memory cleanup.
    """
    
    # Build network
    columns = data['columns']
    net_components = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=False
    )
    
    # Run simulation
    result = run_simulation(
        net_components=net_components,
        neu_exc=neu_exc,
        params={'r_poi': freq * Hz},
        n_trials=n_trials,
        verbose=False
    )
    
    # Extract only essential data
    result_clean = {
        'n_active': result['n_active'],
        'n_spikes': result['n_spikes'],
        'df': result['df'].copy(),
    }
    
    # === Explicit cleanup ===
    del net_components
    del result
    gc.collect()
    
    return (freq, result_clean)

### 4.3 run_batch() function

In [ ]:
# ==================== Cell 6C: 分批运行函数 ====================
def run_batch(freq_list, batch_name, n_workers=3, n_trials=10, checkpoint_dir='./checkpoints/exp1'):
    """
    Run batch with checkpoint and memory monitoring.
    """
    from joblib import Parallel, delayed
    
    print("\n" + "=" * 70)
    print(f"Batch: {batch_name}")
    print("=" * 70)
    
    # Initialize checkpoint
    ckpt = CheckpointManager(checkpoint_dir)
    
    # Check remaining tasks
    remaining = ckpt.get_remaining(freq_list)
    
    if not remaining:
        print(f"✅ All frequencies already completed, loading from checkpoint...")
        return ckpt.load_all_completed()
    
    print(f"\nConfiguration:")
    print(f"  Frequencies: {remaining}")
    print(f"  Trials/freq: {n_trials}")
    print(f"  Workers: {n_workers}")
    print(f"  Checkpoint: {checkpoint_dir}")
    
    # Memory check
    is_safe, msg = check_memory_safe(threshold=80.0, swap_threshold=0.5)
    print(f"\n  Memory status: {msg}")
    
    if not is_safe:
        print(f"  ⚠️ Memory not safe, recommend cleanup first")
        memory_cleanup()
        print_memory("    After cleanup: ")
    
    # Prepare tasks
    tasks = [(freq, NEU_SUGAR_LEFT, DATA, DEFAULT_PARAMS, n_trials) 
             for freq in remaining]
    
    print(f"\n  Running {len(tasks)} tasks...")
    
    with MemoryMonitor(label=batch_name):
        t0 = time()
        
        results = Parallel(n_jobs=n_workers, verbose=5)(
            delayed(run_freq_worker_clean)(*task) for task in tasks
        )
        
        batch_time = time() - t0
    
    # Save to checkpoint
    for freq, result in results:
        ckpt.save(freq, result)
    
    print(f"\n  ✅ Batch complete in {batch_time/60:.2f} min")
    print_memory("  Final memory: ")
    
    # Cleanup
    del results
    gc.collect()
    
    return ckpt.load_all_completed()

### 4.4 Batch Simulation

In [ ]:
# ==================== Cell 7A: Batch 1（频率10-50）====================
print("\n" + "=" * 70)
print("Experiment 1: Batched Execution (3 workers, 4 batches)")
print("=" * 70)

print("\nBatch configuration (4 batches × 5 frequencies):")
for name, freqs in BATCH_CONFIG.items():
    print(f"  {name}: {freqs}")

print("\n⚠️ Run each batch in separate session, restart kernel between batches")
print("=" * 70)

# Run Batch 1
results_batch1 = run_batch(
    freq_list=BATCH_CONFIG['batch1'],
    batch_name='Batch 1',
    n_workers=3,
    n_trials=10
)

print("\n" + "=" * 70)
print("✅ Batch 1 complete!")
print("=" * 70)
print("\n⚠️ NEXT STEP:")
print("1. Check memory status above")
print("2. Kernel → Restart Kernel")
print("3. Re-run Cells 1-6C (setup)")
print("4. Run Cell 7B (Batch 2)")

In [ ]:
# ==================== Cell 7B: Batch 2（频率60-100）====================
# ⚠️ Before running: Kernel → Restart Kernel
# ⚠️ Re-run Cells 1-6C first!

print("\n" + "=" * 70)
print("Running Batch 2...")
print("=" * 70)

results_batch2 = run_batch(
    freq_list=BATCH_CONFIG['batch2'],
    batch_name='Batch 2',
    n_workers=3,
    n_trials=10
)

print("\n✅ Batch 2 complete!")
print("\n⚠️ NEXT: Restart kernel → Run Cells 1-6C → Run Cell 7C")

In [ ]:
# ==================== Cell 7C: Batch 3（频率110-150）====================
# ⚠️ Before running: Kernel → Restart Kernel  
# ⚠️ Re-run Cells 1-6C first!

print("\n" + "=" * 70)
print("Running Batch 3...")
print("=" * 70)

results_batch3 = run_batch(
    freq_list=BATCH_CONFIG['batch3'],
    batch_name='Batch 3',
    n_workers=3,
    n_trials=10
)

print("\n✅ Batch 3 complete!")
print("\n⚠️ NEXT: Restart kernel → Run Cells 1-6C → Run Cell 7D")

In [ ]:
# ==================== Cell 7D: Batch 4（频率160-200）====================
# ⚠️ Before running: Kernel → Restart Kernel
# ⚠️ Re-run Cells 1-6C first!

print("\n" + "=" * 70)
print("Running Batch 4 (Final)...")
print("=" * 70)

results_batch4 = run_batch(
    freq_list=BATCH_CONFIG['batch4'],
    batch_name='Batch 4',
    n_workers=3,
    n_trials=10
)

print("\n✅ Batch 4 complete!")
print("\n⚠️ NEXT: Run Cell 8 (Merge all batches)")

### 4.5 Merge all batches

In [ ]:
# ==================== Cell 8: 合并所有批次结果 ====================
print("\n" + "=" * 70)
print("Merging All Batches")
print("=" * 70)

# Load from checkpoint
ckpt = CheckpointManager('./checkpoints/exp1')
all_results = ckpt.load_all_completed()

print(f"\n✅ Loaded {len(all_results)} frequencies from checkpoint")

# Organize results
results_dict = all_results

# Create summary
summary = []
for freq in sorted(all_results.keys()):
    res = all_results[freq]
    summary.append({
        'Frequency': freq,
        'Active Neurons': res['n_active'],
        'Total Spikes': res['n_spikes'],
    })

df_summary = pd.DataFrame(summary)

## 5 Analysis & Visualization

### 5.1 Calculate firing rates

In [ ]:
# ==================== Cell 9: 计算firing rates ====================

print("=" * 70)
print("Section 4: Results Analysis")
print("=" * 70)

# Extract firing rates for all neurons
print("\n[1/4] Extracting firing rates...")

all_firing_rates = {}
duration_s = float(DEFAULT_PARAMS['t_run'] / ms) / 1000

for freq, res in results_dict.items():
    df = res['df']
    if len(df) > 0:
        spike_counts = df.groupby('flywire_id').size()
        firing_rates = spike_counts / (10 * duration_s)  # 10 trials
        all_firing_rates[freq] = firing_rates.to_dict()
    else:
        all_firing_rates[freq] = {}

print(f"   ✅ Processed {len(all_firing_rates)} frequencies")

# Summary statistics
print(f"\n[2/4] Summary statistics...")
print(df_summary.to_string(index=False))

In [ ]:
# ==================== 新cell：检查最大firing rate ====================

# 方法1：从all_firing_rates检查（如果已计算）
if 'all_firing_rates' in dir():
    max_rate = 0
    max_neuron = None
    max_freq = None
    
    for freq, rates_dict in all_firing_rates.items():
        for neuron_id, rate in rates_dict.items():
            if rate > max_rate:
                max_rate = rate
                max_neuron = neuron_id
                max_freq = freq
    
    print(f"从all_firing_rates:")
    print(f"  最大firing rate: {max_rate:.2f} Hz")
    print(f"  神经元ID: {max_neuron}")
    print(f"  频率: {max_freq} Hz")

# 方法2：直接从results_dict检查
print(f"\n从results_dict直接计算:")

duration_s = 1.0  # 1秒
n_trials = 10

for freq in sorted(results_dict.keys()):
    df = results_dict[freq]['df']
    
    if len(df) > 0:
        # 计算每个神经元的firing rate
        counts = df.groupby('flywire_id').size()
        rates = counts / (n_trials * duration_s)
        
        max_in_freq = rates.max()
        max_neuron_in_freq = rates.idxmax()
        
        print(f"  {freq:3d}Hz: max={max_in_freq:6.2f}Hz, neuron={max_neuron_in_freq}")

### 5.2 Load original data & correlation

In [ ]:
# =====================================================================
# Section 5.2: Efficient Comparison with Original (拆分为3个cells)
# =====================================================================

# ==================== Cell 5.2A: 初步对比，识别差异 ====================

print("\n" + "=" * 70)
print("[5.2A] Initial Comparison - Identify Discrepancies")
print("=" * 70)

# Load original data (v630)
orig_data_path = BASE_DIR / 'Drosophila_brain_model/results/example/sugarR_100Hz.parquet'

if not orig_data_path.exists():
    print(f"\n⚠️ Original data not found: {orig_data_path}")
    has_original = False
else:
    print(f"\nLoading original data (v630)...")
    
    pf = pq.ParquetFile(orig_data_path)
    df_orig_v630 = pd.DataFrame({
        'flywire_id': pf.read(['flywire_id']).column('flywire_id').to_pylist(),
    })
    
    # Calculate original firing rates (v630 IDs)
    spike_counts_v630 = df_orig_v630.groupby('flywire_id').size()
    rate_orig_v630 = spike_counts_v630 / 30.0
    rate_orig_v630.name = 'Original_v630'
    
    # Our results (v783 IDs)
    rate_ours_v783 = pd.Series(all_firing_rates[100], name='Ours_v783')
    
    print(f"   ✅ Loaded")
    print(f"      Original (v630): {len(rate_orig_v630)} neurons")
    print(f"      Ours (v783):     {(rate_ours_v783>0).sum()} neurons")
    
    # === Find discrepancies ===
    print(f"\n[Discrepancy Analysis]")
    
    # Type 1: Only in original (possible ID version change)
    ids_only_orig = set(rate_orig_v630.index) - set(rate_ours_v783.index)
    print(f"   Only in original: {len(ids_only_orig)} neurons")
    
    # Type 2: Only in ours (new in v783)
    ids_only_ours = set(rate_ours_v783[rate_ours_v783>0].index) - set(rate_orig_v630.index)
    print(f"   Only in ours:     {len(ids_only_ours)} neurons")
    
    # Type 3: Common but large firing rate difference
    common_ids = set(rate_orig_v630.index) & set(rate_ours_v783.index)
    df_common = pd.DataFrame({
        'orig': rate_orig_v630[list(common_ids)],
        'ours': rate_ours_v783[list(common_ids)]
    }).fillna(0)
    
    # Active in both
    active_both = (df_common['orig'] > 0) & (df_common['ours'] > 0)
    df_active_common = df_common[active_both].copy()
    df_active_common['abs_diff'] = np.abs(df_active_common['orig'] - df_active_common['ours'])
    
    # Large difference (>20 Hz or >50% relative)
    large_diff_mask = (df_active_common['abs_diff'] > 20) | \
                      (df_active_common['abs_diff'] / df_active_common['orig'] > 0.5)
    ids_large_diff = df_active_common[large_diff_mask].index.tolist()
    
    print(f"   Large differences: {len(ids_large_diff)} neurons (>20Hz or >50%)")
    
    # === Combine all discrepancy types ===
    ids_need_conversion = list(ids_only_orig) + ids_large_diff
    
    print(f"\n[Summary]")
    print(f"   Total IDs to convert: {len(ids_need_conversion)}")
    print(f"   Breakdown:")
    print(f"     - Missing from our data: {len(ids_only_orig)}")
    print(f"     - Large firing rate diff: {len(ids_large_diff)}")
    
    # Show examples
    if ids_only_orig:
        print(f"\n   Examples (missing IDs): {list(ids_only_orig)[:5]}")
    
    if ids_large_diff:
        top_diff = df_active_common.nlargest(5, 'abs_diff')
        print(f"\n   Top 5 firing rate discrepancies:")
        for idx, row in top_diff.iterrows():
            print(f"     {idx}: orig={row['orig']:.1f}, ours={row['ours']:.1f}, diff={row['abs_diff']:.1f}")
    
    has_original = True

In [ ]:
# ==================== Cell 5.2B: 转换差异IDs（精准版）====================
if has_original and 'ids_need_conversion' in dir() and len(ids_need_conversion) > 0:
    print("\n" + "=" * 70)
    print(f"[5.2B] Converting {len(ids_need_conversion)} Discrepancy IDs")
    print("=" * 70)
    
    cache_file_disc = cache_dir / 'discrepancy_ids_v630_to_v783.pkl'
    
    t0 = time()
    
    # Convert only necessary IDs
    ids_converted_v783 = convert_neuron_list_cached(
        old_list=ids_need_conversion,
        cache_file=cache_file_disc,
        new_ver=783,
        verbose=True
    )
    
    conversion_time = time() - t0
    
    print(f"\n✅ Conversion complete in {conversion_time:.1f}s")
    
    # Create mapping (v630 → v783)
    id_mapping = {old: new for old, new in zip(ids_need_conversion, ids_converted_v783)}
    
    # Check how many actually changed
    n_changed = sum(1 for old, new in id_mapping.items() if old != new)
    n_unchanged = len(id_mapping) - n_changed
    
    print(f"\n   ID changes: {n_changed} changed, {n_unchanged} unchanged")
    
    if n_changed > 0:
        print(f"\n   Changed examples (first 5):")
        count = 0
        for old, new in id_mapping.items():
            if old != new and count < 5:
                print(f"     {old} → {new}")
                count += 1
    
else:
    print("\n⚠️ No discrepancies to convert")
    id_mapping = {}

In [ ]:
# ==================== Cell 5.2C: 重新对比（使用v783统一版本）====================
if has_original and 'id_mapping' in dir():
    print("\n" + "=" * 70)
    print("[5.2C] Re-comparison with v783 IDs")
    print("=" * 70)
    
    # 重新加载完整原文数据
    print("\nRe-loading full original spike data...")
    df_orig_full = pd.DataFrame({
        't': pf.read(['t']).column('t').to_pylist(),
        'trial': pf.read(['trial']).column('trial').to_pylist(),
        'flywire_id': pf.read(['flywire_id']).column('flywire_id').to_pylist(),
    })
    
    # 应用ID映射（只替换差异ID）
    if len(id_mapping) > 0:
        print(f"   Applying ID mapping ({len(id_mapping)} IDs)...")
        df_orig_full['flywire_id'] = df_orig_full['flywire_id'].map(
            lambda x: id_mapping.get(x, x)
        )
        n_remapped = df_orig_full['flywire_id'].isin(id_mapping.values()).sum()
        print(f"   Remapped {n_remapped:,} spikes")
    
    # 重新计算firing rate（现在都是v783）
    spike_counts_orig = df_orig_full.groupby('flywire_id').size()
    rate_orig = spike_counts_orig / 30.0
    rate_orig.name = 'Original_v783'
    
    # Our results (already v783)
    rate_ours = pd.Series(all_firing_rates[100], name='Ours_v783')
    
    # === Correlation comparison ===
    print(f"\n[Correlation Analysis]")
    
    # Before conversion (for reference)
    df_compare_old = pd.concat([rate_orig_v630, rate_ours], axis=1).fillna(0)
    r_old, _ = pearsonr(df_compare_old.iloc[:,0], df_compare_old.iloc[:,1])
    
    # After conversion
    df_compare = pd.concat([rate_orig, rate_ours], axis=1).fillna(0)
    r, p = pearsonr(df_compare['Original_v783'], df_compare['Ours_v783'])
    
    print(f"\n  Before ID conversion: r = {r_old:.4f} (v630 vs v783)")
    print(f"  After ID conversion:  r = {r:.4f} (v783 vs v783)")
    print(f"  Improvement:          Δr = {r - r_old:+.4f}")
    
    if r > r_old + 0.01:
        print(f"  ✅ ID conversion improved correlation!")
    else:
        print(f"  ✓ ID version has minimal impact")
    
    # Active neuron counts
    print(f"\n  Active neurons:")
    print(f"    Original: {(df_compare['Original_v783']>0).sum()}")
    print(f"    Ours:     {(df_compare['Ours_v783']>0).sum()}")
    
    # Top neurons overlap
    top_n = 50
    top_orig = df_compare['Original_v783'].nlargest(top_n).index
    top_ours = df_compare['Ours_v783'].nlargest(top_n).index
    overlap = len(set(top_orig) & set(top_ours))
    
    print(f"\n  Top {top_n} overlap: {overlap}/{top_n} ({100*overlap/top_n:.0f}%)")
    
    # Final assessment
    if r > 0.85:
        print(f"\n  ✅ Excellent correlation (r > 0.85)")
    elif r > 0.80:
        print(f"  ✓ Very good (r > 0.80)")
    else:
        print(f"  ✓ Good (r > 0.75)")
    
    print("\n" + "=" * 70)
    
else:
    print("\n⚠️ Skipping comparison")
    has_original = False

### 5.3 Generate plots (4 figures)

In [ ]:

# ==================== Cell 11: 可视化（自动+可定制）====================
print("\n" + "=" * 70)
print("[4/4] Visualization")
print("=" * 70)

# Create output directory
output_dir = Path('./results/exp1_full')
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ==================== Cell 11: 可视化（自动+可定制）====================
print("\n" + "=" * 70)
print("[4/4] Visualization")
print("=" * 70)

# Create output directory
output_dir = Path('./results/exp1_full')
output_dir.mkdir(parents=True, exist_ok=True)

# Plot 1: Correlation (if original data available)
if has_original:
    print("\nPlot 1: Correlation scatter")
    fig1, ax1, r_val = plot_correlation(
        rate_orig, rate_ours, 
        n_trials_ours=10, 
        freq=100,
        # save_path=output_dir / 'correlation_100Hz.png'
    )
    plt.show()

In [ ]:

# Plot 2: Response heatmap
print("\nPlot 2: Response heatmap")
fig2, ax2, top_neurons = plot_response_heatmap(
    all_firing_rates, 
    freq_list=sorted(results_dict.keys()),
    # vmax=199, 
    cmap = sns.color_palette("Spectral_r", as_cmap=True),
    top_n=None,  # Auto-detect
    preset='auto',  # Data-driven
    # save_path=output_dir / 'response_heatmap.png'
)

plt.show()

print(f"   Showing top {len(top_neurons)} neurons")

In [ ]:
# Plot 3: MN9 response curve (if MN9 defined)
if 'NEU_MN9_RIGHT' in dir() and NEU_MN9_RIGHT:
    print("\nPlot 3: MN9 frequency response (bilateral)")
    
    # 合并同侧和对侧
    target_neurons = NEU_MN9_LEFT + NEU_MN9_RIGHT
    labels = ['MN9 Ipsilateral', 'MN9 Contralateral']
    
    fig3, ax3 = plot_frequency_response_curve(
        all_firing_rates,
        figsize=(4, 4),
        freq_list=sorted(results_dict.keys()),
        target_neurons=target_neurons,
        labels=labels,
        save_path=output_dir / 'mn9_bilateral_response.png'
    )
    plt.show()

In [ ]:
# Plot 4: Summary statistics
print("\nPlot 4: Summary statistics")
fig4, axes4 = plot_summary_statistics(
    df_summary,
    save_path=output_dir / 'summary_stats.png'
)
plt.show()

print(f"\n✅ All plots saved to: {output_dir}")

### 5.4 Save complete results

In [ ]:
# ==================== Cell 12: 保存完整结果 ====================
print("\n" + "=" * 70)
print("Saving Results")
print("=" * 70)

# Save complete results
results_package = {
    'results_dict': results_dict,
    'all_firing_rates': all_firing_rates,
    'df_summary': df_summary,
    'parameters': {
        'freq_list': sorted(results_dict.keys()),
        'n_trials': 10,
        'n_frequencies': len(results_dict),
        'neurons_activated': len(NEU_SUGAR_LEFT),
    },
    'validation': {
        'correlation_100Hz': r if has_original else None,
        'top_neurons': top_neurons,
    }
}

# Save
with open(output_dir / 'exp1_complete_results.pkl', 'wb') as f:
    pickle.dump(results_package, f)

# Save summary CSV
df_summary.to_csv(output_dir / 'summary.csv', index=False)

# Save firing rate matrix
if all_firing_rates:
    # Create DataFrame (neurons × frequencies)
    all_freqs = sorted(results_dict.keys())
    all_neurons = set()
    for rates_dict in all_firing_rates.values():
        all_neurons.update(rates_dict.keys())
    
    rate_matrix = pd.DataFrame(index=sorted(all_neurons), columns=all_freqs)
    for freq in all_freqs:
        for neuron_id, rate in all_firing_rates[freq].items():
            rate_matrix.loc[neuron_id, freq] = rate
    
    rate_matrix.fillna(0).to_csv(output_dir / 'firing_rate_matrix.csv')
    print(f"   ✓ firing_rate_matrix.csv ({rate_matrix.shape[0]} neurons × {rate_matrix.shape[1]} freqs)")

print(f"\n✅ Complete results saved:")
print(f"   - exp1_complete_results.pkl (full data)")
print(f"   - summary.csv (table)")
print(f"   - firing_rate_matrix.csv (neurons × frequencies)")
print(f"   - 4 figures (.png)")

print("\n" + "=" * 70)
print("✅ Experiment 1 Complete!")
print("=" * 70)
print(f"\nKey results:")
print(f"  Frequencies tested: {len(results_dict)}")
print(f"  Total active neurons: {len(all_neurons)}")
if has_original:
    print(f"  Correlation with paper: r = {r:.4f}")
print(f"\nOutput: {output_dir}/")

## 6 Save Top 200 neurons

In [ ]:
# ==================== Cell 13: Save Top 200 neurons ====================
print("\n" + "=" * 70)
print("Saving Top 200 Responsive Neurons")
print("=" * 70)

# 按200Hz firing rate排序（与原论文一致）
if 200 in all_firing_rates:
    rates_200hz = pd.Series(all_firing_rates[200])
    top_200_neurons = rates_200hz.sort_values(ascending=False).head(200).index.tolist()
    
    print(f"\nTop 200 neurons (by 200Hz response):")
    print(f"  Count: {len(top_200_neurons)}")
    print(f"  Min rate: {rates_200hz[top_200_neurons[-1]]:.2f} Hz")
    print(f"  Max rate: {rates_200hz[top_200_neurons[0]]:.2f} Hz")
    
    # 保存（供Exp2使用）
    np.save(output_dir / 'top_200_neurons.npy', top_200_neurons)
    
    # 也保存为文本（可读）
    with open(output_dir / 'top_200_neurons.txt', 'w') as f:
        f.write("# Top 200 neurons by 200Hz firing rate\n")
        f.write("# For Exp2 sufficiency test\n")
        for i, nid in enumerate(top_200_neurons, 1):
            f.write(f"{i:3d}. {nid} ({rates_200hz[nid]:.2f} Hz)\n")
    
    print(f"\n✅ Saved:")
    print(f"   - top_200_neurons.npy (for code)")
    print(f"   - top_200_neurons.txt (readable)")
    
else:
    print("\n⚠️ No 200Hz data, cannot select Top 200")

# Exp2 Sufficiency Test
## 7  Sufficiency Test

**Purpose**
Test which neurons can activate MN9 when individually stimulated.

Configuration
- **Neurons**: Top 20 (for quick test)
- **Frequencies**: [50, 100] Hz
- **Trials**: 1 (match baseline)
- **Method**: Parallel execution (3 workers)

Expected Runtime
- Baseline (unoptimized): ~9.6 minutes
- Optimized (parallel): ~2-3 minutes
- Speedup: ~4×

---
run section 1-3, 4.5, 5.1, 5.3, 6 first
### 7.2 Import & Configuration

In [ ]:
# ==================== Cell 7.2: Exp2配置（5批×40神经元） ====================

from flylif.core.experiments import run_exp2_parallel
from flylif.utils.checkpoint import CheckpointManager
from flylif.utils.memory_utils import MemoryMonitor, print_memory, check_memory_safe
import gc

print("=" * 70)
print("Experiment 2: Sufficiency Test - 5 Batches")
print("=" * 70)

# Split 200 neurons into 5 batches (40 each)
BATCH_CONFIG_EXP2 = {
    'batch1': (0, 40),      # Neurons 1-40
    'batch2': (40, 80),     # Neurons 41-80
    'batch3': (80, 120),    # Neurons 81-120
    'batch4': (120, 160),   # Neurons 121-160
    'batch5': (160, 200),   # Neurons 161-200
}

FREQS_EXP2 = [50, 100]
N_TRIALS_EXP2 = 1
TARGET_MN9 = NEU_MN9_RIGHT

print(f"\nConfiguration:")
print(f"  Total neurons: 200 (split into 5 batches of 40)")
print(f"  Frequencies: {FREQS_EXP2}")
print(f"  Trials/condition: {N_TRIALS_EXP2}")
print(f"  Workers: 3")
print(f"  Tasks/batch: 40×2 = 80")
print(f"\nProjected time:")
print(f"  Per batch: ~3 minutes")
print(f"  Total (5 batches): ~15 minutes")

# Initialize checkpoint
checkpoint_dir = Path('./checkpoints/exp2_test')
ckpt_exp2 = CheckpointManager(checkpoint_dir)

print(f"\nCheckpoint: {checkpoint_dir}")
print_memory("\nInitial memory: ")

### 7.3 Batch simulations

In [ ]:
# ==================== Cell 7.3A: Exp2 Batch 1 ====================

print("\n" + "=" * 70)
print("Exp2 Batch 1: Neurons 1-40")
print("=" * 70)

# Check if already completed
batch_key = 'batch1'
if ckpt_exp2.is_completed(batch_key):
    print(f"✅ Batch 1 already completed, loading...")
    results_exp2_batch1 = ckpt_exp2.load(batch_key)
else:
    start, end = BATCH_CONFIG_EXP2['batch1']
    neurons_batch1 = top_200_neurons[start:end]
    
    print(f"Neurons: {start+1}-{end}")
    print(f"Tasks: {len(neurons_batch1) * len(FREQS_EXP2)}")
    
    t0 = time()
    
    with MemoryMonitor(label="Batch1"):
        results_exp2_batch1 = run_exp2_parallel(
            data=DATA,
            neurons_to_test=neurons_batch1,
            freqs=FREQS_EXP2,
            target_neurons=TARGET_MN9,
            params=DEFAULT_PARAMS,
            n_trials=N_TRIALS_EXP2,
            n_workers=3,
            verbose=True
        )
    
    batch1_time = time() - t0
    print(f"\n⏱️  Batch 1 time: {batch1_time/60:.1f} min")
    
    # Save checkpoint
    ckpt_exp2.save(batch_key, results_exp2_batch1)
    print(f"✅ Saved to checkpoint")

print_memory("\nAfter Batch 1: ")

# Check memory before proceeding
is_safe, msg = check_memory_safe(threshold=75.0)
if not is_safe:
    print(f"\n⚠️  {msg}")
    print("   Recommend: Kernel → Restart before Batch 2")
else:
    print(f"\n✅ Memory OK, can proceed to Batch 2")

In [ ]:
# ==================== Cell 7.3B: Exp2 Batch 2 ====================
# ⚠️  如果内存高，先 Restart Kernel → Re-run Cells 1-6 → 再运行此cell

print("\n" + "=" * 70)
print("Exp2 Batch 2: Neurons 41-80")
print("=" * 70)

batch_key = 'batch2'
if ckpt_exp2.is_completed(batch_key):
    print(f"✅ Already completed")
    results_exp2_batch2 = ckpt_exp2.load(batch_key)
else:
    start, end = BATCH_CONFIG_EXP2['batch2']
    neurons_batch2 = top_200_neurons[start:end]
    
    print(f"Neurons: {start+1}-{end}")
    
    t0 = time()
    with MemoryMonitor(label="Batch2"):
        results_exp2_batch2 = run_exp2_parallel(
            data=DATA,
            neurons_to_test=neurons_batch2,
            freqs=FREQS_EXP2,
            target_neurons=TARGET_MN9,
            params=DEFAULT_PARAMS,
            n_trials=N_TRIALS_EXP2,
            n_workers=3,
            verbose=True
        )
    
    print(f"\n⏱️  Batch 2 time: {(time()-t0)/60:.1f} min")
    ckpt_exp2.save(batch_key, results_exp2_batch2)

print_memory("\nAfter Batch 2: ")

In [ ]:
# ==================== Cell 7.3C: Exp2 Batch 3 ====================
# ⚠️  Check memory first

print("\n" + "=" * 70)
print("Exp2 Batch 3: Neurons 81-120 (Testing 4 workers)")
print("=" * 70)

batch_key = 'batch3'
if ckpt_exp2.is_completed(batch_key):
    results_exp2_batch3 = ckpt_exp2.load(batch_key)
else:
    start, end = BATCH_CONFIG_EXP2['batch3']
    neurons_batch3 = top_200_neurons[start:end]

    print(f"Neurons: {start+1}-{end}")
    print(f"Workers: 4 (increased from 3)")

    t0 = time()
    with MemoryMonitor(label="Batch3(4w)"):
        results_exp2_batch3 = run_exp2_parallel(
            DATA, neurons_batch3, FREQS_EXP2, TARGET_MN9,
            DEFAULT_PARAMS, N_TRIALS_EXP2, n_workers=4, verbose=True
        )
    
    print(f"\n⏱️  {(time()-t0)/60:.1f} min")
    ckpt_exp2.save(batch_key, results_exp2_batch3)

print_memory("\nAfter Batch 3: ")

In [ ]:
# ==================== Cell 7.3D: Exp2 Batch 4 (4 workers) ====================

print("\n" + "=" * 70)
print("Exp2 Batch 4: Neurons 121-160")
print("=" * 70)

batch_key = 'batch4'
if ckpt_exp2.is_completed(batch_key):
    results_exp2_batch4 = ckpt_exp2.load(batch_key)
    print("✅ Already completed")
else:
    start, end = BATCH_CONFIG_EXP2['batch4']
    neurons_batch4 = top_200_neurons[start:end]
    
    print(f"Neurons: {start+1}-{end}")
    print(f"Workers: 4")
    
    t0 = time()
    with MemoryMonitor(label="Batch4"):
        results_exp2_batch4 = run_exp2_parallel(
            DATA, neurons_batch4, FREQS_EXP2, TARGET_MN9,
            DEFAULT_PARAMS, N_TRIALS_EXP2, 
            n_workers=4,  # ← 4 workers
            verbose=True
        )
    
    print(f"\n⏱️  {(time()-t0)/60:.1f} min")
    ckpt_exp2.save(batch_key, results_exp2_batch4)

print_memory("\nAfter Batch 4: ")

In [ ]:
# ==================== Cell 7.3E: Exp2 Batch 5 - Final (4 workers) ====================

print("\n" + "=" * 70)
print("Exp2 Batch 5 (Final): Neurons 161-200")
print("=" * 70)

batch_key = 'batch5'
if ckpt_exp2.is_completed(batch_key):
    results_exp2_batch5 = ckpt_exp2.load(batch_key)
    print("✅ Already completed")
else:
    start, end = BATCH_CONFIG_EXP2['batch5']
    neurons_batch5 = top_200_neurons[start:end]
    
    print(f"Neurons: {start+1}-{end}")
    print(f"Workers: 4")
    
    t0 = time()
    with MemoryMonitor(label="Batch5"):
        results_exp2_batch5 = run_exp2_parallel(
            DATA, neurons_batch5, FREQS_EXP2, TARGET_MN9,
            DEFAULT_PARAMS, N_TRIALS_EXP2, 
            n_workers=4,
            verbose=True
        )
    
    print(f"\n⏱️  {(time()-t0)/60:.1f} min")
    ckpt_exp2.save(batch_key, results_exp2_batch5)

print_memory("\nAfter Batch 5: ")
print("\n" + "=" * 70)
print("✅ All 5 Exp2 Batches Complete!")
print("=" * 70)

---

### 7.4 merge all batches

In [ ]:
# ==================== Cell 7.4: 合并所有Exp2批次 ====================

print("\n" + "=" * 70)
print("Merging All Exp2 Batches")
print("=" * 70)

# Load all from checkpoint
all_batches = []
for batch_key in ['batch1', 'batch2', 'batch3', 'batch4', 'batch5']:
    batch_result = ckpt_exp2.load(batch_key)
    if batch_result:
        all_batches.append(batch_result)
    else:
        print(f"⚠️  {batch_key} not found")

print(f"Loaded {len(all_batches)} batches")

# Merge
results_exp2 = {}
for freq in FREQS_EXP2:
    results_exp2[freq] = {}
    for batch in all_batches:
        results_exp2[freq].update(batch[freq])

print(f"\n✅ Merged results:")
for freq in FREQS_EXP2:
    print(f"  {freq} Hz: {len(results_exp2[freq])} neurons")

# Validation (same as Cell 7.5)
for freq in FREQS_EXP2:
    mn9_means = [stats['mean'] for stats in results_exp2[freq].values()]
    n_activate = sum(1 for m in mn9_means if m > 0)
    
    print(f"\n{freq} Hz:")
    print(f"  Neurons activating MN9: {n_activate}/200")
    print(f"  MN9 range: {min(mn9_means):.1f} - {max(mn9_means):.1f} Hz")

---

### 7.5 Results Validation

In [ ]:
# ==================== Cell 7.5: Exp2可视化（保持Exp1顺序） ====================

from flylif.utils.visualization import plot_response_heatmap

print("\n" + "=" * 70)
print("Exp2 Visualization")
print("=" * 70)

# 数据转换
all_firing_rates_exp2 = {}
for freq in FREQS_EXP2:
    all_firing_rates_exp2[freq] = {
        nid: stats['mean'] 
        for nid, stats in results_exp2[freq].items()
    }

# Plot（保持Exp1的神经元顺序）
print("Generating heatmap (matching Exp1 neuron order)...")

fig, ax, _ = plot_response_heatmap(
    all_firing_rates=all_firing_rates_exp2,
    freq_list=FREQS_EXP2,  # 不sorted
    neuron_order=top_200_neurons,  # ← 关键：使用Exp1顺序
    # vmax=199, 
    cmap = sns.color_palette("Spectral_r", as_cmap=True),
    # save_path='./results/exp2_test/exp2_heatmap.png'
)

# 修改labels（Exp2特定）
ax.set_xlabel('Activation Firing Rate (Hz)', fontsize=12)
ax.set_ylabel('Neurons Tested (ordered by sugar response)', fontsize=12)
ax.set_title('Exp2: Predicted MN9 Activation\n(Individual Neuron Stimulation)', 
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Heatmap generated (neuron order matches Exp1)")

In [ ]:
# ===== Plot 2: Summary Bar Chart =====
print(f"\n[2/3] Generating summary statistics...")

# Count neurons that activate MN9 at each frequency
summary_data = []
for freq in sorted(FREQS_EXP2):
    mn9_rates = [stats['mean'] for stats in results_exp2[freq].values()]
    n_activate = sum(1 for r in mn9_rates if r > 0)
    max_rate = max(mn9_rates)
    mean_rate = np.mean([r for r in mn9_rates if r > 0]) if n_activate > 0 else 0
    
    summary_data.append({
        'Frequency': freq,
        'N_Activate_MN9': n_activate,
        'Max_MN9_Rate': max_rate,
        'Mean_MN9_Rate': mean_rate,
    })

df_summary_exp2 = pd.DataFrame(summary_data)

# Plot
fig2, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Count
ax2a = axes[0]
bars = ax2a.bar(df_summary_exp2['Frequency'], df_summary_exp2['N_Activate_MN9'],
                color='steelblue', edgecolor='black', alpha=0.8, width=15)
ax2a.set_xlabel('Activation Frequency (Hz)', fontsize=11)
ax2a.set_ylabel('Neurons Activating MN9 (>0 Hz)', fontsize=11)
ax2a.set_title('Sufficient Neurons vs Frequency', fontsize=12, fontweight='bold')
ax2a.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax2a.text(bar.get_x() + bar.get_width()/2., height,
              f'{int(height)}', ha='center', va='bottom', fontsize=10)

# Panel B: Mean MN9 rate
ax2b = axes[1]
ax2b.bar(df_summary_exp2['Frequency'], df_summary_exp2['Mean_MN9_Rate'],
         color='coral', edgecolor='black', alpha=0.8, width=15)
ax2b.set_xlabel('Activation Frequency (Hz)', fontsize=11)
ax2b.set_ylabel('Mean MN9 Firing Rate (Hz)', fontsize=11)
ax2b.set_title('Mean MN9 Response (Active Neurons)', fontsize=12, fontweight='bold')
ax2b.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"  ✅ Summary charts generated")

In [ ]:
# ===== Plot 3: Top Neurons Response Curves =====
print(f"\n[3/3] Generating top neurons response curves...")

# Get top 10 neurons by max MN9 activation
all_max_rates = {}
for nid in results_exp2[FREQS_EXP2[0]].keys():
    max_rate = max(results_exp2[freq][nid]['mean'] for freq in FREQS_EXP2)
    all_max_rates[nid] = max_rate

top_10_neurons = sorted(all_max_rates.items(), key=lambda x: x[1], reverse=True)[:10]

fig3, ax3 = plt.subplots(figsize=(10, 6))

for i, (nid, max_rate) in enumerate(top_10_neurons):
    rates = [results_exp2[freq][nid]['mean'] for freq in sorted(FREQS_EXP2)]
    stds = [results_exp2[freq][nid]['std'] for freq in sorted(FREQS_EXP2)]
    
    ax3.errorbar(sorted(FREQS_EXP2), rates, yerr=stds,
                 marker='o', markersize=6, capsize=4,
                 linewidth=1.5, alpha=0.7, 
                 label=f'Top {i+1} ({max_rate:.1f} Hz max)',
                 )

ax3.set_xlabel('Activation Frequency (Hz)', fontsize=12)
ax3.set_ylabel('Predicted MN9 Firing Rate (Hz)', fontsize=12)
ax3.set_title('Top 10 Neurons: MN9 Activation Dose-Response', 
              fontsize=13, fontweight='bold')
ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
# plt.savefig(output_dir / 'exp2_top10_response_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"  ✅ Response curves generated")

In [ ]:
# ===== Summary Table =====
print(f"\n" + "=" * 70)
print("Exp2 Summary Statistics")
print("=" * 70)
print(df_summary_exp2.to_string(index=False))

print(f"\nTop 10 neurons by max MN9 activation:")
for i, (nid, max_rate) in enumerate(top_10_neurons, 1):
    rates_str = ", ".join([f"{results_exp2[f][nid]['mean']:.1f}" for f in sorted(FREQS_EXP2)])
    print(f"  {i:2d}. {nid}: [{rates_str}] Hz")

---

### 7.6 Save Results

In [ ]:
# ==================== Cell 7.6: 保存Exp2完整结果（修复版） ====================

output_dir = Path('./results/exp2_test')
output_dir.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)
print("Saving Exp2 Results")
print("=" * 70)

# ===== 计算summary statistics =====
summary_data = []
for freq in FREQS_EXP2:
    mn9_rates = [stats['mean'] for stats in results_exp2[freq].values()]
    n_activate = sum(1 for r in mn9_rates if r > 0)
    
    summary_data.append({
        'Frequency': freq,
        'N_Activate_MN9': n_activate,
        'N_Total': len(results_exp2[freq]),
        'Percent_Active': f"{100*n_activate/len(results_exp2[freq]):.1f}%",
        'Max_MN9_Hz': max(mn9_rates),
        'Mean_MN9_Hz': np.mean([r for r in mn9_rates if r > 0]) if n_activate > 0 else 0,
    })

df_summary_exp2 = pd.DataFrame(summary_data)

# ===== 保存完整结果 =====
exp2_package = {
    'neurons_tested': list(top_200_neurons),  # ← 修复：兼容list和array
    'freqs': FREQS_EXP2,
    'n_trials': N_TRIALS_EXP2,
    'target_neurons': TARGET_MN9,  # ← 修复：变量名
    'results': results_exp2,
    'summary': df_summary_exp2.to_dict(),
    'parameters': {
        'n_batches': 5,
        'neurons_per_batch': 40,
        'n_workers': 4,
    },
    'notes': 'Exp2 sufficiency test: 200 neurons × 2 frequencies × 1 trial'
}

with open(output_dir / 'exp2_results_200neurons.pkl', 'wb') as f:
    pickle.dump(exp2_package, f)

# 保存summary CSV
df_summary_exp2.to_csv(output_dir / 'exp2_summary.csv', index=False)

# 保存firing rate matrix
rate_matrix_exp2 = pd.DataFrame(
    {freq: [results_exp2[freq][nid]['mean'] for nid in top_200_neurons]
     for freq in FREQS_EXP2},
    index=top_200_neurons
)
rate_matrix_exp2.to_csv(output_dir / 'exp2_firing_rate_matrix.csv')

print(f"\n✅ Results saved:")
print(f"   {output_dir}/exp2_results_200neurons.pkl")
print(f"   {output_dir}/exp2_summary.csv")
print(f"   {output_dir}/exp2_firing_rate_matrix.csv")

print(f"\n" + "=" * 70)
print("Exp2 Summary")
print("=" * 70)
print(df_summary_exp2.to_string(index=False))